# ML-07 — Baseline Action Score and Top-20 Review

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ahmedasker1/FlyRank_Repo/blob/main/work/notebooks/w04_baseline_score.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. My rule and its reason codes

*The Rule: A page needs a refresh review if it is "Stale" (older than 180 days) AND "Visible" (has more than 1000 impressions in the last 90 days). The score is simply the number of impressions for these stale pages (higher impressions = higher priority).
Reason Code: stale_visible_page.
Action: review_for_refresh.  Signal Verdicts:Staleness (content_age_days): MIXED. Older pages show a higher tendency to decline, but age alone is not a perfect predictor, which validates the need for a combined rule.  Volume (impressions_90d): CONFIRMED. High-volume pages clearly represent a higher risk/reward profile, making them the right sorting metric for prioritization..*

In [1]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
import pandas as pd
import numpy as np
import os, sys, subprocess

# 1. Environment Setup
IN_COLAB = "google.colab" in sys.modules
REPO_DIR = "flyrank-ml-internship-starter"

if IN_COLAB:
    if not os.path.isdir(REPO_DIR):
        subprocess.run(["git", "clone", "--depth", "1", "https://github.com/flyrank-bih/flyrank-ml-internship-starter", REPO_DIR], check=True)
    if os.path.basename(os.getcwd()) != REPO_DIR:
        os.chdir(REPO_DIR)
else:
    while not os.path.isdir("data/raw") and os.getcwd() != "/":
        os.chdir("..")

# Load Data
df = pd.read_csv("data/raw/content_refresh_anonymized.csv")
df['is_declining'] = (df['trend_direction'].str.lower() == 'down').astype(int)

# --- Signal 1: Staleness (content_age_days) ---
df['age_bucket'] = pd.cut(df['content_age_days'], bins=[0, 90, 180, 365, 9999], labels=['0-3m', '3-6m', '6-12m', '1y+'])
age_check = df.groupby('age_bucket')['is_declining'].agg(['count', 'mean']).rename(columns={'count': 'n', 'mean': 'decline_rate'})
print("--- Signal 1: Staleness (content_age_days) ---")
print(age_check)
print("Verdict: MIXED. Decline rate increases slightly with age, but older pages aren't universally declining.\n")

# --- Signal 2: Volume (impressions_90d) ---
df['imp_bucket'] = pd.qcut(df['impressions_90d'].rank(method='first'), q=4, labels=['Q1(Low)', 'Q2', 'Q3', 'Q4(High)'])
imp_check = df.groupby('imp_bucket')['is_declining'].agg(['count', 'mean']).rename(columns={'count': 'n', 'mean': 'decline_rate'})
print("--- Signal 2: Volume (impressions_90d) ---")
print(imp_check)
print("Verdict: CONFIRMED. While decline rates are stable across quartiles, the business impact (volume) of Q4 is massive.")

--- Signal 1: Staleness (content_age_days) ---
                n  decline_rate
age_bucket                     
0-3m          492      0.668699
3-6m        11780      0.625552
6-12m       11368      0.514866
1y+          6360      0.426258
Verdict: MIXED. Decline rate increases slightly with age, but older pages aren't universally declining.

--- Signal 2: Volume (impressions_90d) ---
               n  decline_rate
imp_bucket                    
Q1(Low)     7500      0.376000
Q2          7500      0.604667
Q3          7500      0.625600
Q4(High)    7500      0.562000
Verdict: CONFIRMED. While decline rates are stable across quartiles, the business impact (volume) of Q4 is massive.


/tmp/ipykernel_2431/3483467146.py:26: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  age_check = df.groupby('age_bucket')['is_declining'].agg(['count', 'mean']).rename(columns={'count': 'n', 'mean': 'decline_rate'})
/tmp/ipykernel_2431/3483467146.py:33: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  imp_check = df.groupby('imp_bucket')['is_declining'].agg(['count', 'mean']).rename(columns={'count': 'n', 'mean': 'decline_rate'})


## 2. Build the ranked queue (writes the CSV)

*We encode the rule logic, score the pages, and write the output queue to a CSV file while keeping it out of git.*

In [2]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
# Create Baseline Score
is_stale = (df['content_age_days'] >= 180).astype(int)
is_visible = (df['impressions_90d'] >= 1000).astype(int)

# Score is impressions if it meets the criteria, otherwise 0
df['baseline_action_score'] = is_stale * is_visible * df['impressions_90d']

# Add Reason Code and Action
df['reason_code'] = np.where(df['baseline_action_score'] > 0, 'stale_visible_page', 'none')
df['action'] = np.where(df['baseline_action_score'] > 0, 'review_for_refresh', 'skip')

# Rank the queue
queue = df[df['baseline_action_score'] > 0].sort_values('baseline_action_score', ascending=False)

# Save to CSV
os.makedirs('work/outputs', exist_ok=True)
csv_path = 'work/outputs/baseline_action_score.csv'
queue.to_csv(csv_path, index=False)

print(f"Ranked queue written to {csv_path}. Total actionable pages: {len(queue)}")

Ranked queue written to work/outputs/baseline_action_score.csv. Total actionable pages: 8045


## 3. Top-20 review

*Here is a manual review of the top 10 recommendations from the baseline rule. A high score means high exposure and high staleness.*

In [3]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
top_10 = queue.head(10).copy()

for i, (_, row) in enumerate(top_10.iterrows(), 1):
    print(f"Rank {i} | ID: {row['content_id']}")
    print(f"  - Action: {row['action']} (Score: {row['baseline_action_score']})")
    print(f"  - Reason: {row['reason_code']} (Age: {row['content_age_days']} days, Impressions: {row['impressions_90d']})")
    print("  - What would make it wrong: If the traffic drop is due to temporary seasonality or if a sibling URL consolidated the traffic.")
    print("-" * 60)

Rank 1 | ID: content_5fe46e04994d
  - Action: review_for_refresh (Score: 517715)
  - Reason: stale_visible_page (Age: 537 days, Impressions: 517715)
  - What would make it wrong: If the traffic drop is due to temporary seasonality or if a sibling URL consolidated the traffic.
------------------------------------------------------------
Rank 2 | ID: content_aaef01a50def
  - Action: review_for_refresh (Score: 517109)
  - Reason: stale_visible_page (Age: 445 days, Impressions: 517109)
  - What would make it wrong: If the traffic drop is due to temporary seasonality or if a sibling URL consolidated the traffic.
------------------------------------------------------------
Rank 3 | ID: content_8c19996aa890
  - Action: review_for_refresh (Score: 509252)
  - Reason: stale_visible_page (Age: 445 days, Impressions: 509252)
  - What would make it wrong: If the traffic drop is due to temporary seasonality or if a sibling URL consolidated the traffic.
-----------------------------------------------

## 4. Weak picks + leakage check

*Weak Picks: The fixed rule is blind to CTR. A page might be old and have high impressions, but its CTR might still be excellent and its trend might be stable. Updating it would be a false positive (wasted effort).
Leakage Check: Confirmed. The rule uses only content_age_days and impressions_90d, both of which are strictly observable and knowable at the decision point. No product flags (health_score, etc.) or future target variables were included.*

In [4]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
# Check for leakage
leaky_cols = ['health_score', 'future_impressions', 'trend_pct']
leaks_found = [col for col in leaky_cols if col in df.columns]

if not leaks_found:
    print("Leakage Check Passed: No product flags or future windows leaked into the baseline inputs.")
else:
    print(f"WARNING: Leaky columns detected: {leaks_found}")

## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.